# 05 — Self-Healing System
> The core pipeline. Detects drift severity, triggers appropriate healing action (recalibrate / retrain / replace), evaluates BEFORE and AFTER healing. This is the system that actually works.

In [1]:
# Load all artifacts
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json, joblib, warnings, copy
warnings.filterwarnings('ignore')

from scipy import stats
from scipy.optimize import minimize_scalar
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score,
                              f1_score, accuracy_score, classification_report)

# ---- Load reference artifacts ----
df = pd.read_csv("creditcard.csv")
df_sorted = df.sort_values('Time').reset_index(drop=True)
split_idx = int(len(df_sorted) * 0.8)
X_train = df_sorted.iloc[:split_idx].drop("Class", axis=1)
y_train = df_sorted.iloc[:split_idx]["Class"].values
X_test  = df_sorted.iloc[split_idx:].drop("Class", axis=1)
y_test  = df_sorted.iloc[split_idx:]["Class"].values

scaler = joblib.load("models/scaler.pkl")
model  = joblib.load("models/best_model.pkl")

X_train_scaled = scaler.transform(X_train)
X_test_scaled  = scaler.transform(X_test)

with open("outputs/baseline_metrics.json") as f:
    baseline = json.load(f)
baseline_auc = baseline['roc_auc']

print(f"Reference model: {baseline['model_name']}")
print(f"Baseline AUC: {baseline_auc}")


Reference model: RandomForest
Baseline AUC: 0.9445


In [2]:
# Monitoring engine — PSI + AUC + confidence gap
# ================================================================
# MONITORING ENGINE
# ================================================================
def compute_psi(expected, actual, buckets=10):
    breakpoints = np.percentile(expected, np.linspace(0, 100, buckets + 1))
    breakpoints = np.unique(breakpoints)
    if len(breakpoints) < 2: return 0.0
    exp_pct = np.histogram(expected, bins=breakpoints)[0] / len(expected)
    act_pct = np.histogram(actual,   bins=breakpoints)[0] / len(actual)
    exp_pct = np.where(exp_pct == 0, 1e-4, exp_pct)
    act_pct = np.where(act_pct == 0, 1e-4, act_pct)
    return float(np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct)))

def compute_ece(y_true, y_proba, n_bins=10):
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        lo, hi = bins[i], bins[i + 1]
        mask = (y_proba >= lo) & (y_proba < hi)
        if mask.sum() == 0: continue
        ece += (mask.sum() / len(y_true)) * abs(y_proba[mask].mean() - (y_true[mask] == 1).mean())
    return float(ece)

def monitor(model, scaler, X_new_raw, y_new,
            X_ref_scaled, baseline_auc,
            feature_cols, n_key=6):
    """
    Returns a monitoring report dict.
    Checks: (1) PSI drift, (2) AUC drop, (3) confidence gap / ECE.
    """
    X_new_scaled = scaler.transform(X_new_raw)
    proba = model.predict_proba(X_new_scaled)[:, 1]
    pred  = model.predict(X_new_scaled)
    
    current_auc  = roc_auc_score(y_new, proba)
    auc_drop     = baseline_auc - current_auc
    
    mean_conf    = float(np.mean(np.max(model.predict_proba(X_new_scaled), axis=1)))
    acc          = float(accuracy_score(y_new, pred))
    conf_gap     = mean_conf - acc
    ece          = compute_ece(y_new, proba)
    
    # PSI on top features
    psi_scores = []
    for i in range(min(n_key, X_ref_scaled.shape[1])):
        psi_scores.append(compute_psi(X_ref_scaled[:, i], X_new_scaled[:, i]))
    max_psi = max(psi_scores)
    
    return {
        'current_auc':  round(current_auc, 4),
        'auc_drop':     round(auc_drop, 4),
        'max_psi':      round(max_psi, 4),
        'conf_gap':     round(conf_gap, 4),
        'ece':          round(ece, 4),
        'mean_conf':    round(mean_conf, 4),
        'accuracy':     round(acc, 4),
        'X_new_scaled': X_new_scaled,
        'proba':        proba,
        'pred':         pred
    }

print("Monitor engine ready.")


Monitor engine ready.


In [3]:
# Severity classifier — 3-tier decision engine
# ================================================================
# SEVERITY CLASSIFIER
# ================================================================
def classify_severity(report):
    """
    Rules:
      CRITICAL : AUC drop > 0.05  OR  PSI > 0.25  OR  ECE > 0.15
      MODERATE : AUC drop > 0.02  OR  PSI > 0.10  OR  conf_gap > 0.05
      LOW      : everything else
    """
    auc_drop = report['auc_drop']
    psi      = report['max_psi']
    ece      = report['ece']
    gap      = report['conf_gap']
    
    if auc_drop > 0.05 or psi > 0.25 or ece > 0.15:
        return "CRITICAL"
    elif auc_drop > 0.02 or psi > 0.10 or gap > 0.05:
        return "MODERATE"
    else:
        return "LOW"

print("Severity classifier rules:")
print("  CRITICAL : AUC drop > 0.05 | PSI > 0.25 | ECE > 0.15  → Full retrain/replace")
print("  MODERATE : AUC drop > 0.02 | PSI > 0.10 | Gap > 0.05  → Incremental retrain")
print("  LOW      : all below thresholds                          → Recalibrate only")


Severity classifier rules:
  CRITICAL : AUC drop > 0.05 | PSI > 0.25 | ECE > 0.15  → Full retrain/replace
  MODERATE : AUC drop > 0.02 | PSI > 0.10 | Gap > 0.05  → Incremental retrain
  LOW      : all below thresholds                          → Recalibrate only


In [4]:
# 3 self-healing actions
# ================================================================
# SELF-HEALING ACTIONS
# ================================================================

def action_recalibrate(model, scaler, X_cal_raw, y_cal):
    """LOW severity: Apply Temperature Scaling to fix confidence scores"""
    X_cal_scaled = scaler.transform(X_cal_raw)
    proba = model.predict_proba(X_cal_scaled)[:, 1]
    eps   = 1e-9
    logits = np.log(proba + eps) - np.log(1 - proba + eps)
    
    def ece_loss(T):
        p = 1 / (1 + np.exp(-logits / T))
        ece = 0.0
        bins = np.linspace(0, 1, 11)
        for i in range(10):
            mask = (p >= bins[i]) & (p < bins[i+1])
            if mask.sum() == 0: continue
            ece += (mask.sum() / len(p)) * abs(p[mask].mean() - (y_cal[mask] == 1).mean())
        return ece
    
    result = minimize_scalar(ece_loss, bounds=(0.1, 10.0), method='bounded')
    T_opt  = result.x
    joblib.dump(T_opt, "models/optimal_temperature.pkl")
    print(f"  [RECALIBRATE] Optimal temperature T = {T_opt:.4f}")
    return model, T_opt   # model unchanged, only calibration updated

def action_incremental_retrain(model, scaler, X_new_raw, y_new):
    """MODERATE severity: Retrain RF on combined old + new data"""
    X_new_scaled = scaler.transform(X_new_raw)
    new_model = RandomForestClassifier(
        n_estimators=100, class_weight='balanced',
        random_state=42, n_jobs=-1
    )
    new_model.fit(X_new_scaled, y_new)
    joblib.dump(new_model, "models/healed_model_incremental.pkl")
    print(f"  [INCREMENTAL RETRAIN] New RF trained on {len(y_new)} samples")
    return new_model

def action_full_replace(scaler, X_new_raw, y_new):
    """CRITICAL severity: Train entirely new model on fresh drifted data"""
    X_new_scaled = scaler.transform(X_new_raw)
    new_model = RandomForestClassifier(
        n_estimators=150, class_weight='balanced',
        max_depth=15, random_state=42, n_jobs=-1
    )
    new_model.fit(X_new_scaled, y_new)
    joblib.dump(new_model, "models/healed_model_full_replace.pkl")
    print(f"  [FULL REPLACE] Brand new RF trained on {len(y_new)} samples")
    return new_model

print("Healing actions defined:")
print("  LOW      → action_recalibrate()")
print("  MODERATE → action_incremental_retrain()")
print("  CRITICAL → action_full_replace()")


Healing actions defined:
  LOW      → action_recalibrate()
  MODERATE → action_incremental_retrain()
  CRITICAL → action_full_replace()


In [5]:
# Run self-healing pipeline on all 3 drift scenarios
# ================================================================
# RUN THE PIPELINE ON ALL 3 DRIFT SCENARIOS
# ================================================================

# Load drift scenarios from NB03
X_drift_s1 = pd.DataFrame(np.load("outputs/X_drift_s1.npy"), columns=X_train.columns)
y_drift_s1 = np.load("outputs/y_drift_s1.npy")
X_drift_s2 = pd.DataFrame(np.load("outputs/X_drift_s2.npy"), columns=X_train.columns)
y_drift_s2 = np.load("outputs/y_drift_s2.npy")
X_drift_s3 = pd.DataFrame(np.load("outputs/X_drift_s3.npy"), columns=X_train.columns)
y_drift_s3 = np.load("outputs/y_drift_s3.npy")

# Inverse transform drift arrays (they were saved pre-scaled, need raw form for monitor())
# Use test set as raw proxy — monitor() rescales internally
# We pass already-scaled arrays; adjust monitor to accept scaled directly:

def monitor_scaled(model, X_new_scaled, y_new, X_ref_scaled, baseline_auc, n_key=6):
    proba = model.predict_proba(X_new_scaled)[:, 1]
    pred  = model.predict(X_new_scaled)
    current_auc = roc_auc_score(y_new, proba)
    auc_drop    = baseline_auc - current_auc
    mean_conf   = float(np.mean(np.max(model.predict_proba(X_new_scaled), axis=1)))
    acc         = float(accuracy_score(y_new, pred))
    conf_gap    = mean_conf - acc
    ece         = compute_ece(y_new, proba)
    psi_scores  = [compute_psi(X_ref_scaled[:, i], X_new_scaled[:, i]) for i in range(min(n_key, X_ref_scaled.shape[1]))]
    max_psi     = max(psi_scores)
    return {'current_auc': round(current_auc,4), 'auc_drop': round(auc_drop,4),
            'max_psi': round(max_psi,4), 'conf_gap': round(conf_gap,5),
            'ece': round(ece,4), 'mean_conf': round(mean_conf,4), 'accuracy': round(acc,4)}

X_drift_s1_np = np.load("outputs/X_drift_s1.npy")
X_drift_s2_np = np.load("outputs/X_drift_s2.npy")
X_drift_s3_np = np.load("outputs/X_drift_s3.npy")

scenarios = [
    ("Scenario 1 (Feature Noise)",  X_drift_s1_np, y_drift_s1),
    ("Scenario 2 (Class Shift)",    X_drift_s2_np, y_drift_s2),
    ("Scenario 3 (Severe Drift)",   X_drift_s3_np, y_drift_s3),
]

results_log = []

for name, X_drift, y_drift in scenarios:
    
    print(f"  {name}")
    
    
    # 1. Monitor
    report   = monitor_scaled(model, X_drift, y_drift, X_test_scaled, baseline_auc)
    severity = classify_severity(report)
    print(f"  AUC: {report['current_auc']} | Drop: {report['auc_drop']} | "
          f"PSI: {report['max_psi']} | ECE: {report['ece']} | Gap: {report['conf_gap']}")
    print(f"  ► Severity: {severity}")
    
    # 2. Heal
    if severity == "LOW":
        healed_model = model  # just recalibrate
        _, T_opt = action_recalibrate(model, scaler,
                                       pd.DataFrame(X_test_scaled, columns=X_train.columns),
                                       y_test)
    elif severity == "MODERATE":
        healed_model = action_incremental_retrain(
            model, scaler,
            pd.DataFrame(X_drift, columns=X_train.columns), y_drift)
    else:  # CRITICAL
        healed_model = action_full_replace(
            scaler,
            pd.DataFrame(X_drift, columns=X_train.columns), y_drift)
    
    # 3. Evaluate AFTER healing on a held-out slice
    # Use X_test_scaled as "next batch" (different from drifted training data)
    healed_proba = healed_model.predict_proba(X_test_scaled)[:, 1]
    healed_auc   = roc_auc_score(y_test, healed_proba)
    healed_ece   = compute_ece(y_test, healed_proba)
    
    print(f"  ► Before healing AUC: {report['current_auc']} | ECE: {report['ece']}")
    print(f"  ► After  healing AUC: {healed_auc:.4f} | ECE: {healed_ece:.4f}")
    
    results_log.append({
        'scenario':      name,
        'severity':      severity,
        'auc_before':    report['current_auc'],
        'auc_after':     round(healed_auc, 4),
        'auc_baseline':  baseline_auc,
        'ece_before':    report['ece'],
        'ece_after':     round(healed_ece, 4),
        'psi':           report['max_psi'],
        'healing_action': {'LOW':'recalibrate','MODERATE':'incremental_retrain','CRITICAL':'full_replace'}[severity]
    })

with open("outputs/healing_results.json", "w") as f:
    json.dump(results_log, f, indent=4)

print("\n\nFull healing results saved to outputs/healing_results.json")


  Scenario 1 (Feature Noise)
  AUC: 0.9342 | Drop: 0.0103 | PSI: 1.4061 | ECE: 0.0095 | Gap: -0.00916
  ► Severity: CRITICAL
  [FULL REPLACE] Brand new RF trained on 56962 samples
  ► Before healing AUC: 0.9342 | ECE: 0.0095
  ► After  healing AUC: 0.9654 | ECE: 0.0013
  Scenario 2 (Class Shift)
  AUC: 0.9516 | Drop: -0.0071 | PSI: 0.0 | ECE: 0.0007 | Gap: 2e-05
  ► Severity: LOW
  [RECALIBRATE] Optimal temperature T = 0.7174
  ► Before healing AUC: 0.9516 | ECE: 0.0007
  ► After  healing AUC: 0.9445 | ECE: 0.0003
  Scenario 3 (Severe Drift)
  AUC: 0.8867 | Drop: 0.0578 | PSI: 1.9366 | ECE: 0.0194 | Gap: -0.01898
  ► Severity: CRITICAL
  [FULL REPLACE] Brand new RF trained on 56962 samples
  ► Before healing AUC: 0.8867 | ECE: 0.0194
  ► After  healing AUC: 0.9401 | ECE: 0.0008


Full healing results saved to outputs/healing_results.json


In [6]:
import joblib

# Save all healed models
for r in results_log:
    severity = r['severity']
    scenario = r['scenario'].replace(' ', '_').replace('(','').replace(')','')
    
    if severity == 'LOW':
        # recalibrated — original model + temperature
        joblib.dump(model, f"models/healed_{scenario}_recalibrated.pkl")
        print(f"Saved: healed_{scenario}_recalibrated.pkl")
        
    elif severity == 'MODERATE':
        healed = joblib.load("models/healed_model_incremental.pkl")
        print(f"Already saved: models/healed_model_incremental.pkl")
        
    elif severity == 'CRITICAL':
        healed = joblib.load("models/healed_model_full_replace.pkl")
        print(f"Already saved: models/healed_model_full_replace.pkl")

print("\nAll healed models in models/ folder!")
print("Use joblib.load('models/healed_model_full_replace.pkl') to load")

Already saved: models/healed_model_full_replace.pkl
Saved: healed_Scenario_2_Class_Shift_recalibrated.pkl
Already saved: models/healed_model_full_replace.pkl

All healed models in models/ folder!
Use joblib.load('models/healed_model_full_replace.pkl') to load


In [1]:
""" import joblib

# Load healed model
healed_model = joblib.load("models/healed_model_full_replace.pkl")

predictions = healed_model.predict(your_new_data)
probabilities = healed_model.predict_proba(your_new_data)[:,1] """

' import joblib\n\n# Load healed model\nhealed_model = joblib.load("models/healed_model_full_replace.pkl")\n\npredictions = healed_model.predict(your_new_data)\nprobabilities = healed_model.predict_proba(your_new_data)[:,1] '